# Assignment 07 - Notebook 04
## Hai dataset: Diabetes và chứng khoán AAPL

Notebook xác nhận hai bộ dữ liệu được nộp cùng repository và giải thích vai trò của từng bộ trong hệ thống Knowledge/RAG Agent.

## 1. Khởi tạo

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / 'knowledge').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('Project:', ROOT)

Project: C:\DATA\assignment3\assignment7


In [2]:
import pandas as pd
from IPython.display import display
diabetes = pd.read_csv(ROOT/'data/diabetes_dataset.csv')
stock = pd.read_csv(ROOT/'data/AAPL_2015_2025.csv', parse_dates=['Date'])
print('Diabetes:', diabetes.shape)
print('AAPL:', stock.shape)

Diabetes: (100000, 16)
AAPL: (2766, 7)


## 2. Dataset Diabetes

Mỗi hàng là một quan sát bệnh nhân. Mười lăm cột đầu vào gồm thông tin nhân khẩu, bệnh nền, BMI, HbA1c và glucose; cột `diabetes` là nhãn. MLP từ Assignment 3 được dùng như một công cụ của Agent.

In [3]:
display(diabetes.head())
print('Phân bố nhãn:')
print(diabetes['diabetes'].value_counts())

   year  gender   age  ... hbA1c_level  blood_glucose_level  diabetes
0  2020  Female  32.0  ...         5.0                  100         0
1  2015  Female  29.0  ...         5.0                   90         0
2  2015    Male  18.0  ...         4.8                  160         0
3  2015    Male  41.0  ...         4.0                  159         0
4  2016  Female  52.0  ...         6.5                   90         0

[5 rows x 16 columns]
Phân bố nhãn:
diabetes
0    91500
1     8500
Name: count, dtype: int64


## 3. Dataset chứng khoán AAPL

Mỗi hàng là một phiên giao dịch, gồm OHLC, giá điều chỉnh và khối lượng. Dữ liệu từ 2015 đến 2025, phù hợp tạo cửa sổ thời gian cho RNN hoặc công cụ tóm tắt xu hướng.

In [4]:
display(stock.head())
print('Khoảng thời gian:', stock['Date'].min().date(), 'đến', stock['Date'].max().date())
print(stock[['Open','High','Low','Close','Volume']].describe().round(2))

        Date       Open       High        Low      Close  Adj Close     Volume
0 2015-01-02  27.847500  27.860001  26.837500  27.332500  24.171757  212818400
1 2015-01-05  27.072500  27.162500  26.352501  26.562500  23.490799  257142000
2 2015-01-06  26.635000  26.857500  26.157499  26.565001  23.493013  263188400
3 2015-01-07  26.799999  27.049999  26.674999  26.937500  23.822439  160423600
4 2015-01-08  27.307501  28.037500  27.174999  27.972500  24.737740  237458000
Khoảng thời gian: 2015-01-02 đến 2025-12-31
          Open     High      Low    Close        Volume
count  2766.00  2766.00  2766.00  2766.00  2.766000e+03
mean    108.55   109.72   107.46   108.64  1.113994e+08
std      73.78    74.56    73.07    73.86  6.804243e+07
min      22.50    22.92    22.37    22.58  1.791060e+07
25%      39.16    39.46    38.77    39.08  6.488962e+07
50%      91.18    92.41    90.28    91.20  9.454380e+07
75%     170.96   172.63   169.35   170.83  1.372937e+08
max     286.20   288.62   283.30  

## 4. Đặc trưng chuỗi thời gian

Return theo ngày và trung bình động chỉ sử dụng dữ liệu quá khứ. Khi train mô hình phải chia theo thời gian, không xáo trộn ngẫu nhiên để tránh rò rỉ tương lai.

In [5]:
stock = stock.sort_values('Date').copy()
stock['Return'] = stock['Close'].pct_change()
stock['MA20'] = stock['Close'].rolling(20).mean()
stock[['Date','Close','Return','MA20']].tail()

           Date       Close    Return        MA20
2761 2025-12-24  273.809998  0.005324  277.151999
2762 2025-12-26  273.399994 -0.001497  276.944499
2763 2025-12-29  273.760010  0.001317  276.689999
2764 2025-12-30  273.079987 -0.002484  276.188998
2765 2025-12-31  271.859985 -0.004468  275.472498

## 5. Vai trò trong Agent

Diabetes phục vụ công cụ ước lượng nhãn sức khỏe; AAPL phục vụ công cụ phân tích chuỗi giá. Knowledge Base/RAG cung cấp phần giải thích bằng văn bản và nguồn. Hai dataset tabular được lưu trong `data/`, còn corpus và graph nằm trong `knowledge/`.